# 함수

In [2]:
# mq_quant.py

import FinanceDataReader as fdr
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as mticker
import warnings
import seaborn as sns
sns.set_theme()


warnings.simplefilter(action='ignore', category=FutureWarning)

def get_close(ticker, start, end=None):
    """
    종가 데이터 수집 (Get Close Prices)
    """
    return fdr.DataReader(ticker, start, end)['Close']

def calc_daily_return(prices):
    """
    일별 수익률 계산 (Calculate Daily Returns)
    """
    return prices.pct_change().fillna(0)

def calc_cum_return(prices):
    """
    누적 수익률(자산 흐름) 계산 (Calculate Cumulative Returns)
    """
    return prices / prices.iloc[0]

def calc_portfolio(prices, weight=None):
    """
    포트폴리오 수익률 계산 (Calculate Portfolio Returns)
    returns: (일별 수익률, 누적 수익률)
    """
    # 1. 누적 수익률 계산
    cum_ret = calc_cum_return(prices)

    # 2. 비중 설정 (기본값: 동일 비중)
    if not weight:
        weight = [1/len(prices.columns)] * len(prices.columns)

    # 3. 포트폴리오 누적 수익률 (가중 평균)
    # 각 자산의 누적 수익률에 비중을 곱해 합산합니다. (Buy & Hold 가정)
    port_cum_ret = (cum_ret * weight).sum(axis=1)

    # 4. 포트폴리오 일별 수익률 (역산)
    port_daily_ret = port_cum_ret.pct_change().fillna(0)

    return port_daily_ret, port_cum_ret


def evaluate_performance(daily_ret, cum_ret, risk_free_rate=0.02):
    """
    성과 지표 평가 (CAGR, MDD, Volatility, Sharpe ratio)
    """

    # 1. 수익성 (CAGR)
    total_ret = cum_ret.iloc[-1]
    years = len(cum_ret) / 252
    cagr = total_ret ** (1 / years) - 1

    # 2. 안정성 (MDD)
    historical_max = cum_ret.cummax()
    dd = (cum_ret - historical_max) / historical_max * 100
    mdd = dd.min()

    # 3. 위험 (변동성)
    daily_vol = daily_ret.std()               # 일일 변동성
    annual_vol = daily_vol * np.sqrt(252)     # 연간 변동성

    # 4. 효율 (Sharpe Ratio)
    annual_return = daily_ret.mean() * 252    # 연간 수익률 (산술 평균)
    sharpe_ratio = (annual_return - risk_free_rate) / annual_vol # 위험 한 단위당 수익

    print(f"▶ 최종 수익률   : {(cum_ret.iloc[-1]-1)*100:.2f}%")
    print(f"=== 성과 평가 리포트 ===")
    print(f"1. 수익성 (CAGR) : {cagr*100:.2f}%")
    print(f"2. 안정성 (MDD)  : {mdd:.2f}%")
    print(f"-" * 30)
    print(f"3. 연간 변동성   : {annual_vol*100:.2f}%")
    print(f"4. 샤프 지수     : {sharpe_ratio:.4f}")

    return cagr, dd, mdd

def get_rebalancing_dates(close_data, period="month"):
    """
    리밸런싱 날짜 추출 (Pandas Grouper 활용)
    - 입력: 종가 데이터, 주기('month', 'quarter', 'year')
    - 출력: 리밸런싱 시행일(DatetimeIndex)
    """
    # 주기별 Pandas Frequency 문자열 매핑
    freq_map = {'month': 'ME', 'quarter': 'QE', 'year': 'YE'}

    # 1. 예외처리: 잘못된 주기 입력
    if period not in freq_map:
        raise ValueError("period must be 'month', 'quarter', or 'year'")

    freq = freq_map[period]

    rebalancing_dates = close_data.groupby(pd.Grouper(freq=freq)).apply(lambda x: x.index[-1])

    return rebalancing_dates.sort_index()


def cal_rebalancing_portfolio(close_data, period="month", weight_df=None, enable_plot=True):
    """
    리밸런싱 포트폴리오 성과 계산 (Chunk 방식)
    """

    # 1. 리밸런싱 날짜 구하기 (전체 데이터 기준)
    rebal_dates = get_rebalancing_dates(close_data, period)

    # 날짜 동기화 (weight_df가 있다면, 교집합 날짜만 사용)
    # close_data가 더 길어도, weight_df가 있는 기간만 백테스팅 수행
    if weight_df is not None:
        # weight_df 인덱스에 포함된 날짜만 필터링
        rebal_dates = rebal_dates[rebal_dates.isin(weight_df.index)]

        # 만약 겹치는 날짜가 하나도 없다면 에러 처리
        if rebal_dates.empty:
            print("Error: close_data와 weight_df의 리밸런싱 날짜가 일치하지 않습니다.")
            return None, None

    # 3. 초기 비중 설정 (없으면 동일 비중)
    if weight_df is None:
        n_assets = len(close_data.columns)
        weight_df = pd.DataFrame(
            index=rebal_dates,
            columns=close_data.columns,
            data=1/n_assets
        )

    # 4. 데이터 범위 보정 (시작일 기준)
    first_date = rebal_dates[0] # 필터링된 첫 날짜


    # 전체 기간 수익률 계산
    full_daily_rets = close_data.pct_change().fillna(0)

    # 백테스트 시작일 이후 데이터만 슬라이싱
    daily_rets = full_daily_rets.loc[first_date:]

    portfolio_chunks = []
    total_value = 1.0

    # 리밸런싱 기간별 순회
    full_dates = list(rebal_dates)

    # 마지막 구간 처리: 마지막 리밸런싱 날짜 ~ 데이터 끝 날짜
    # (단, daily_rets의 마지막 날짜가 리밸런싱 날짜보다 뒤에 있을 때만)
    if full_dates[-1] < daily_rets.index[-1]:
        full_dates.append(daily_rets.index[-1])

    for start, end in zip(full_dates[:-1], full_dates[1:]):

        # start가 weight_df에 없으면 건너뜀 (안전 장치)
        if start not in weight_df.index:
            continue

        current_weights = weight_df.loc[start]

        # start 다음날부터 end까지의 수익률 사용 (start 당일은 리밸런싱 날)
        chunk_rets = daily_rets.loc[start:end].iloc[1:]

        if chunk_rets.empty: continue

        cum_growth = (1 + chunk_rets).cumprod()
        chunk_value = (cum_growth * current_weights).sum(axis=1) * total_value
        portfolio_chunks.append(chunk_value)
        total_value = chunk_value.iloc[-1]

    # 6. 결과 병합
    if not portfolio_chunks:
        return None, None

    portfolio_cum_ret = pd.concat(portfolio_chunks)
    portfolio_cum_ret.loc[first_date] = 1.0 # 시작점 1.0 강제 할당
    portfolio_cum_ret = portfolio_cum_ret.sort_index()

    portfolio_day_ret = portfolio_cum_ret.pct_change().fillna(0)

    if enable_plot:
        historical_max = portfolio_cum_ret.cummax()
        dd = (portfolio_cum_ret - historical_max) / historical_max * 100
        mdd = dd.min()

        # [수정] 전체 컬럼이 아니라, weight_df에 존재하는(실제 투자된) 티커만 추출
        if weight_df is not None:
            # weight_df의 컬럼 중 close_data에도 존재하는 것만 리스트로 만듦 (KeyError 방지)
            tickers = [col for col in weight_df.columns if col in close_data.columns]

            if 'cash' in tickers: tickers.remove('cash')
        else:
            tickers = close_data.columns.tolist()

        plot_daily_rets = full_daily_rets.loc[portfolio_cum_ret.index]

        plot_portfolio_result(tickers, plot_daily_rets, portfolio_cum_ret, dd, mdd)

    return portfolio_day_ret, portfolio_cum_ret


def plot_portfolio_result(tickers, daily_rets, port_cum_ret, dd, mdd):
    """
    포트폴리오 성과 시각화 함수 (백테스트 기간 일치 버전)
    """

    plt.figure(figsize=(10, 8))

    # [차트 1] 포트폴리오 누적 수익률 (Asset Growth)
    plt.subplot(2, 1, 1)

    # 1. 메인 포트폴리오 선 그리기
    plt.plot(port_cum_ret.index, port_cum_ret, label='Portfolio', color='#d62728', linewidth=1.5)

    # 2. 개별 종목 흐름 그리기 (기간 매칭 및 리베이스)
    if daily_rets is not None and tickers is not None:

        # [핵심 수정] 포트폴리오의 시작/종료 날짜 구하기
        start_date = port_cum_ret.index[0]
        end_date = port_cum_ret.index[-1]

        # [핵심 수정] 전체 데이터(daily_rets)를 실제 백테스트 기간만큼만 자르기
        subset_daily_rets = daily_rets.loc[start_date:end_date]

        for t in tickers:
            if t in subset_daily_rets.columns:
                # [핵심 수정] 잘라낸 기간의 수익률로 누적 수익률 새로 계산
                # 이렇게 해야 시작점이 1.0(혹은 포트폴리오 시작점) 근처로 맞춰집니다.
                ind_cum = (1 + subset_daily_rets[t]).cumprod()

                # 시각화 (투명도 조절)
                plt.plot(ind_cum.index, ind_cum, label=f'{t}', alpha=0.3, linewidth=0.8, linestyle='--')

    # Y축 로그 스케일 및 포맷 설정
    plt.yscale('log')
    ax = plt.gca()
    ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.yaxis.set_minor_formatter(mticker.ScalarFormatter())
    ax.ticklabel_format(style='plain', axis='y')

    plt.axhline(1.0, color='gray', linestyle=':', alpha=0.5)
    plt.title('Portfolio Cumulative Return', fontsize=14, fontweight='bold')

    # 범례가 너무 많으면 가리니까, 적절히 위치 조정
    plt.legend(loc='upper left', fontsize='small')
    plt.grid(True, alpha=0.3)


    # [차트 2] 낙폭 (Drawdown)
    plt.subplot(2, 1, 2)

    plt.fill_between(dd.index, dd, 0, color='#1f77b4', alpha=0.3)
    plt.plot(dd.index, dd, color='#1f77b4', linewidth=0.5)

    plt.title('Portfolio Drawdown (MDD)', fontsize=12)
    plt.axhline(mdd, color='red', linestyle='--', label=f'Max DD: {mdd:.2f}%')

    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.ylabel('Drawdown (%)')

    plt.tight_layout()
    plt.show()

def cal_AvgMomentumScore(close_data, n = 12):
    """
    평균 모멘텀 스코어를 기반으로 한 투자 비중 구하기
    close_data: 종가 데이터
    n: 모멘텀 기간 1~n
    return: 투자비중 weight df, 평균모멘텀 스코어 df
    """
    avgMomentumScore = 0 # 평모스 초기값
    priceOnRebalDate = close_data.loc[get_rebalancing_dates(close_data)] # 리밸런싱 일자의 가격 데이터

    # 1 ~ n개월 모멘텀 스코어 합
    for i in range(1, n+1):
        avgMomentumScore = np.where(priceOnRebalDate / priceOnRebalDate.shift(i) > 1, 1, 0) + avgMomentumScore

    # 평모스 계산
    avgMomentumScore = pd.DataFrame(avgMomentumScore, index=priceOnRebalDate.index, columns=priceOnRebalDate.columns) # dataframe 형변환
    avgMomentumScore = avgMomentumScore / n

    # 모멘텀 스코어에 따른 weight 계산
    weight = avgMomentumScore.divide(avgMomentumScore.sum(axis=1), axis=0).fillna(0)
    # 투자 비중이 모두 0인 구간에서는 현금 보유
    weight['cash'] = np.where(weight.sum(axis=1) == 0, 1, 0)

    # 투자비중, 평모스 리턴
    return weight, avgMomentumScore

def get_heatmap(cumReturn):
    """
    cagr, dd, mdd 계산 및 히트맵(연간/월간 수익률) 시각화
    """
    # 1. CAGR 계산
    cagr = cumReturn.iloc[-1] ** (252/len(cumReturn))

    # 2. MDD 계산
    dd = (cumReturn.cummax() - cumReturn) / cumReturn.cummax() * 100
    mdd = dd.max()

    print(f"최종수익률: {cumReturn.iloc[-1]:.4f}\ncagr: {cagr:.4f}\nmdd: {mdd:.4f}%")

    # 3. 데이터프레임 변환 및 전처리
    cumReturn = pd.DataFrame(cumReturn)
    cumReturn.columns = ['cumReturn']
    cumReturn['year'] = cumReturn.index.year
    cumReturn['month'] = cumReturn.index.month

    # 월별/연별 마지막 데이터만 추출
    monthData = cumReturn.drop_duplicates(['year', 'month'], keep="last").copy()
    yearData = cumReturn.drop_duplicates('year', keep="last").copy()

    # 4. 수익률 계산
    monthData['monthReturn'] = monthData['cumReturn'].pct_change().fillna(0) * 100
    # 연간 수익률
    yearData['yearReturn'] = yearData['cumReturn'].pct_change().fillna(0) * 100
    # 첫 해 수익률 보정 (1.0에서 시작했다고 가정)
    yearData.iloc[0, -1] = (yearData.iloc[0, 0] - 1) * 100

    # 시각화를 위해 마지막 데이터의 '월'을 강제로 12로 설정 (연간 수익률 히트맵 정렬용)
    yearData.iloc[-1, yearData.columns.get_loc('month')] = 12

    # 5. 히트맵 시각화
    # 피벗 테이블 생성
    monthPivot = monthData.pivot(index='year', columns='month', values='monthReturn')
    yearPivot = yearData.pivot(index='year', columns='month', values='yearReturn')

    # 그래프 그리기
    # width_ratios=[1, 5]: 연간 수익률(왼쪽)은 좁게, 월간 수익률(오른쪽)은 넓게 설정
    f, (ax1, ax2) = plt.subplots(1, 2, sharey=True, gridspec_kw={'width_ratios': [1, 5]}, figsize=(20, 10))

    # 왼쪽: 연간 수익률
    sns.heatmap(yearPivot, ax=ax1, annot=True, fmt='.2f', linewidths=.3, cmap="RdYlGn", center=0, cbar=False)
    ax1.set_title("Annual Return")

    # 오른쪽: 월간 수익률
    sns.heatmap(monthPivot, ax=ax2, annot=True, fmt='.2f', linewidths=.1, cmap="RdYlGn", center=0, cbar=False)
    ax2.set_title("Monthly Return")

    plt.tight_layout()
    plt.show()

# DAA 모멘텀
- top2

In [ ]:
import pandas as pd

def get_rebalancing_dates(close_data, period="month"):
    """
    리밸런싱 날짜 추출 (Pandas Grouper 활용)
    - 입력: 종가 데이터, 주기('month', 'quarter', 'year')
    - 출력: 리밸런싱 시행일(DatetimeIndex)
    """
    # 주기별 Pandas Frequency 문자열 매핑
    freq_map = {'month': 'ME', 'quarter': 'QE', 'year': 'YE'}

    # 1. 예외처리: 잘못된 주기 입력
    if period not in freq_map:
        raise ValueError("period must be 'month', 'quarter', or 'year'")

    freq = freq_map[period]

    rebalancing_dates = close_data.groupby(pd.Grouper(freq=freq)).apply(lambda x: x.index[-1])

    return rebalancing_dates.sort_index()

def getDAAWeight_case(closeDataSet, case='A'):

    daaCol = ['SMH', 'CQQQ', 'SOXX', 'SPY', 'IEMG', 'EWY','DX-Y.NYB','GLD', 'TLT', 'VWO', 'BND',"LQD"]
    daaAttack = ['SMH', 'CQQQ', 'SOXX', 'SPY', 'IEMG', 'EWY']

    daaDefense = ['DX-Y.NYB', 'GLD', 'TLT']
    
    # Case별 카나리아 자산 설정
    if case == "A":
        daaCanary = ["VWO", "BND"]      # Original
    elif case == "B":
        daaCanary = ["LQD", "BND"]      # 신용 위험 중심
    elif case == "C":
        daaCanary = ["SPY", "BND"]      # 미국 시장 중심
    else:
        daaCanary = ["VWO", "BND"]      # Default

    daaData = closeDataSet[daaCol].copy()
    daaData.dropna(inplace=True)

    # 모멘텀 스코어 계산
    rebalDate = get_rebalancing_dates(daaData)
    daaDataOnRebalDate = daaData.loc[rebalDate]

    momentum1 = daaDataOnRebalDate / daaDataOnRebalDate.shift(1) -1
    momentum3 = daaDataOnRebalDate / daaDataOnRebalDate.shift(3) - 1
    momentum6 = daaDataOnRebalDate / daaDataOnRebalDate.shift(6) -1
    momentum12 = daaDataOnRebalDate / daaDataOnRebalDate.shift(12) -1

    momentumScore = 12*momentum1 + 4*momentum3 + 2*momentum6 + momentum12
    momentumScore.dropna(inplace=True)

    # 카나리아 자산 모멘텀 스코어가 모두 0 초과일 때 공격 자산 중 모멘텀 스코어가 가장 큰 2개 자산을 보유
    # 카나리아 자산 중 하나의 자산만 모멘텀 스코어0 초과일 때 모멘텀 스코어가 가장 큰 공격 자산 1개와 방어자산 1개
    # 카나리아 자산 모두 모멘텀스코어가 0 이하라면 수비 자산 중 모멘텀 스코어가 가장 큰 자산에 몰빵
    isAttack = (momentumScore[daaCanary] > 0).sum(axis=1)
    daaWeight = momentumScore.apply(applyGetDAAWeight, axis=1, args=(isAttack,))
    return daaWeight

def applyGetDAAWeight(row, isAttack):
    daaCol = ['SMH', 'CQQQ', 'SOXX', 'SPY', 'IEMG', 'EWY','DX-Y.NYB','GLD', 'TLT', 'VWO','BND',"LQD"]
    daaAttack = ['SMH', 'CQQQ', 'SOXX', 'SPY', 'IEMG', 'EWY']

    # UST 데이터 시점 문제로 TLT로 대체
    daaDefense = ['DX-Y.NYB', 'GLD', 'TLT']
    daaCanary = ["VWO", "BND"]

    if isAttack[row.name] == 2:
        # 카나리아 자산 모멘텀 스코어가 모두 0 초과일 때 공격 자산 중 모멘텀 스코어가 가장 큰 2개 자산을 보유
        top2 = row[daaAttack].nlargest(n=2).index
        result = pd.Series(row.index.isin(top2), index=row.index, name=row.name).astype(int).replace(1, 0.5)
        return result

    if isAttack[row.name] == 1:
        # 카나리아 자산 중 하나의 자산만 모멘텀 스코어0 초과일 때 모멘텀 스코어가 가장 큰 공격 자산 1개와 방어자산 1개
        topAttack = row[daaAttack].idxmax()
        topDefense = row[daaDefense].idxmax()
        result = pd.Series(row.index.isin([topAttack, topDefense]), index=row.index, name=row.name).astype(int).replace(1, 0.5)
        return result

    # 카나리아 자산 모두 모멘텀스코어가 0 이하라면 수비 자산 중 모멘텀 스코어가 가장 큰 자산에 몰빵
    return pd.Series(row.index == row[daaDefense].idxmax(), index=row.index, name=row.name).astype(int)

# ETF 유니버스.csv 필요

In [38]:
import os
current_path = os.path.dirname(os.path.abspath('ETF 유니버스.csv'))
current_path 

'c:\\Users\\cloud\\Desktop\\AI 퀀트 과정\\project2\\project2'

In [39]:
import pandas as pd
import os

current_path = os.path.dirname(os.path.abspath('ETF 유니버스.csv'))
file_path = os.path.join(current_path, 'data', 'ETF 유니버스.csv')

close_data = pd.read_csv(file_path, index_col=0, parse_dates=True)

In [40]:
close_data.head()

,SPY,QQQ,IWM,DIA,VUG,IWO,VTV,IWN,USMV,QUAL,...,BLOK,BIL,SGOV,NEAR,MINT,ICSH,JPST,PULS,DX-Y.NYB,VNQ.1
Date,,,,,,,,,,,,,,,,,,,,,
2005-01-03,81.61,33.70,48.28,67.29,40.93,56.88,31.28,42.25,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,81.30,22.79
2005-01-04,80.61,33.08,47.24,66.65,40.56,55.44,31.00,41.48,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,82.57,22.45
2005-01-05,80.05,32.88,46.30,66.28,40.40,54.48,30.86,40.65,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,82.54,21.70
2005-01-06,80.46,32.71,46.54,66.49,40.39,54.74,31.01,40.76,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.15,21.87
2005-01-07,80.34,32.88,46.02,66.36,40.30,54.14,30.95,40.32,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.61,21.82


In [41]:
daaWeight = getDAAWeight_case(close_data, 'A').loc["2019-12-31":]
daaDayReturn, daaCumReturn_a = cal_rebalancing_portfolio(close_data=close_data, weight_df=daaWeight, enable_plot=False)
daaCagr, daaDD_a, daaMDD = evaluate_performance(daaDayReturn, daaCumReturn_a)

▶ 최종 수익률   : 169.52%
=== 성과 평가 리포트 ===
1. 수익성 (CAGR) : 17.83%
2. 안정성 (MDD)  : -26.71%
------------------------------
3. 연간 변동성   : 22.66%
4. 샤프 지수     : 0.7494


In [42]:
daaWeight = getDAAWeight_case(close_data, 'B').loc["2019-12-31":]
daaDayReturn, daaCumReturn_a = cal_rebalancing_portfolio(close_data=close_data, weight_df=daaWeight, enable_plot=False)
daaCagr, daaDD_a, daaMDD = evaluate_performance(daaDayReturn, daaCumReturn_a)

▶ 최종 수익률   : 142.45%
=== 성과 평가 리포트 ===
1. 수익성 (CAGR) : 15.78%
2. 안정성 (MDD)  : -31.53%
------------------------------
3. 연간 변동성   : 24.98%
4. 샤프 지수     : 0.6325


In [43]:
daaWeight = getDAAWeight_case(close_data, 'C').loc["2019-12-31":]
daaDayReturn, daaCumReturn_a = cal_rebalancing_portfolio(close_data=close_data, weight_df=daaWeight, enable_plot=False)
daaCagr, daaDD_a, daaMDD = evaluate_performance(daaDayReturn, daaCumReturn_a)

▶ 최종 수익률   : 174.60%
=== 성과 평가 리포트 ===
1. 수익성 (CAGR) : 18.19%
2. 안정성 (MDD)  : -26.86%
------------------------------
3. 연간 변동성   : 23.97%
4. 샤프 지수     : 0.7341


# 1. 단순 동일 비중

In [109]:
def getDAAWeight_case(closeDataSet, case='A',  show_momentum_details=False):

    daaCol = ['SMH', 'CQQQ', 'SOXX', 'SPY', 'IEMG', 'EWY','DX-Y.NYB','GLD', 'TLT', 'VWO', 'BND',"LQD"]
    daaAttack = ['SMH', 'CQQQ', 'SOXX', 'SPY', 'IEMG', 'EWY']

    daaDefense = ['DX-Y.NYB', 'GLD', 'TLT']
    
    # Case별 카나리아 자산 설정
    if case == "A":
        daaCanary = ["VWO", "BND"]      # Original
    elif case == "B":
        daaCanary = ["LQD", "BND"]      # 신용 위험 중심
    elif case == "C":
        daaCanary = ["SPY", "BND"]      # 미국 시장 중심
    else:
        daaCanary = ["VWO", "BND"]      # Default

    daaData = closeDataSet[daaCol].copy()
    daaData.dropna(inplace=True)

    # 모멘텀 스코어 계산
    rebalDate = get_rebalancing_dates(daaData)
    daaDataOnRebalDate = daaData.loc[rebalDate]

    momentum1 = daaDataOnRebalDate / daaDataOnRebalDate.shift(1) -1
    momentum3 = daaDataOnRebalDate / daaDataOnRebalDate.shift(3) - 1
    momentum6 = daaDataOnRebalDate / daaDataOnRebalDate.shift(6) -1
    momentum12 = daaDataOnRebalDate / daaDataOnRebalDate.shift(12) -1

    momentumScore = 12*momentum1 + 4*momentum3 + 2*momentum6 + momentum12
    # ========================================================================
    # 5. 모멘텀 상세 정보 출력 (옵션)
    # ========================================================================
    if show_momentum_details:
        print("\n" + "="*70)
        print("모멘텀 스코어 계산 공식:")
        print("최종 스코어 = 12 × (1개월 수익률) + 4 × (3개월 수익률) + 2 × (6개월 수익률) + 1 × (12개월 수익률)")
        print("="*70)
        
        print("\n[1개월 모멘텀 (가중치: 12)]")
        print(momentum1.tail(3))
        
        print("\n[3개월 모멘텀 (가중치: 4)]")
        print(momentum3.tail(3))
        
        print("\n[6개월 모멘텀 (가중치: 2)]")
        print(momentum6.tail(3))
        
        print("\n[12개월 모멘텀 (가중치: 1)]")
        print(momentum12.tail(3))
        
        print("\n[최종 가중 모멘텀 스코어]")
        print(momentumScore.tail(3))
        print("="*70 + "\n")
    
    # 결측치 제거
    momentumScore.dropna(inplace=True)

    # 카나리아 자산 모멘텀 스코어가 모두 0 초과일 때 공격자산 6개 전체에 균등배분 (각 16.67%)
    # 카나리아 자산 중 하나의 자산만 모멘텀 스코어0 초과일 때 모멘텀 스코어가 가장 큰 공격 자산 1개와 방어자산 1개
    # 카나리아 자산 모두 모멘텀스코어가 0 이하라면 수비 자산 중 모멘텀 스코어가 가장 큰 자산에 몰빵
    isAttack = (momentumScore[daaCanary] > 0).sum(axis=1)
    daaWeight = momentumScore.apply(applyGetDAAWeight_by_same_w, axis=1, args=(isAttack,))
    return daaWeight

def applyGetDAAWeight_by_same_w(row, isAttack):
    daaCol = ['SMH', 'CQQQ', 'SOXX', 'SPY', 'IEMG', 'EWY','DX-Y.NYB','GLD', 'TLT', 'VWO','BND',"LQD"]
    daaAttack = ['SMH', 'CQQQ', 'SOXX', 'SPY', 'IEMG', 'EWY']
    daaDefense = ['DX-Y.NYB', 'GLD', 'TLT']
    daaCanary = ["VWO", "BND"]
    
    # 카나리아 2개 살아있을 때: 공격자산 전체를 동일비중으로 보유
    if isAttack[row.name] == 2:
        attack_weight = 1.0 / len(daaAttack)
        result = pd.Series(row.index.isin(daaAttack), index=row.index, name=row.name).astype(float)
        result = result.replace(1.0, attack_weight)
        return result
    
    # 카나리아 1개 살아있을 때: 공격자산 50%, 방어자산 50%
    if isAttack[row.name] == 1:
        attack_weight = 0.5 / len(daaAttack)
        defense_weight = 0.5 / len(daaDefense)
        
        result = pd.Series(0.0, index=row.index, name=row.name)
        result[daaAttack] = attack_weight
        result[daaDefense] = defense_weight
        return result
    
    # 카나리아 0개: 방어자산 전체를 동일비중으로 보유
    defense_weight = 1.0 / len(daaDefense)
    result = pd.Series(row.index.isin(daaDefense), index=row.index, name=row.name).astype(float)
    result = result.replace(1.0, defense_weight)
    return result

## 카나리
"A":    daaCanary = ["VWO", "BND"]      # Original

"B":    daaCanary = ["LQD", "BND"]      # 신용 위험 중심

"C":    daaCanary = ["SPY", "BND"]      # 미국 시장 중심

In [110]:
daaWeight = getDAAWeight_case(close_data, 'A').loc["2019-12-31":]
daaDayReturn, daaCumReturn_a = cal_rebalancing_portfolio(close_data=close_data, weight_df=daaWeight, enable_plot=False)
daaCagr, daaDD_a, daaMDD = evaluate_performance(daaDayReturn, daaCumReturn_a)

▶ 최종 수익률   : 98.63%
=== 성과 평가 리포트 ===
1. 수익성 (CAGR) : 12.02%
2. 안정성 (MDD)  : -25.54%
------------------------------
3. 연간 변동성   : 17.95%
4. 샤프 지수     : 0.6107


In [111]:
daaWeight = getDAAWeight_case(close_data, 'B').loc["2019-12-31":]
daaDayReturn, daaCumReturn_a = cal_rebalancing_portfolio(close_data=close_data, weight_df=daaWeight, enable_plot=False)
daaCagr, daaDD_a, daaMDD = evaluate_performance(daaDayReturn, daaCumReturn_a)

▶ 최종 수익률   : 87.53%
=== 성과 평가 리포트 ===
1. 수익성 (CAGR) : 10.96%
2. 안정성 (MDD)  : -31.08%
------------------------------
3. 연간 변동성   : 20.72%
4. 샤프 지수     : 0.5098


In [112]:
daaWeight = getDAAWeight_case(close_data, 'C').loc["2019-12-31":]
daaDayReturn, daaCumReturn_a = cal_rebalancing_portfolio(close_data=close_data, weight_df=daaWeight, enable_plot=False)
daaCagr, daaDD_a, daaMDD = evaluate_performance(daaDayReturn, daaCumReturn_a)

▶ 최종 수익률   : 108.53%
=== 성과 평가 리포트 ===
1. 수익성 (CAGR) : 12.93%
2. 안정성 (MDD)  : -25.84%
------------------------------
3. 연간 변동성   : 18.10%
4. 샤프 지수     : 0.6521


# 2. weighted Average

In [ ]:
def getDAAWeight_wa(closeDataSet, case='A',  show_momentum_details=False):

    daaCol = ['SMH', 'CQQQ', 'SOXX', 'SPY', 'IEMG', 'EWY','DX-Y.NYB','GLD', 'TLT', 'VWO', 'BND',"LQD"]
    daaAttack = ['SMH', 'CQQQ', 'SOXX', 'SPY', 'IEMG', 'EWY']

    daaDefense = ['DX-Y.NYB', 'GLD', 'TLT']
    
    # Case별 카나리아 자산 설정
    if case == "A":
        daaCanary = ["VWO", "BND"]      # Original
    elif case == "B":
        daaCanary = ["LQD", "BND"]      # 신용 위험 중심
    elif case == "C":
        daaCanary = ["SPY", "BND"]      # 미국 시장 중심
    else:
        daaCanary = ["VWO", "BND"]      # Default

    daaData = closeDataSet[daaCol].copy()
    daaData.dropna(inplace=True)

    # 모멘텀 스코어 계산
    rebalDate = get_rebalancing_dates(daaData)
    daaDataOnRebalDate = daaData.loc[rebalDate]

    momentum1 = daaDataOnRebalDate / daaDataOnRebalDate.shift(1) -1
    momentum3 = daaDataOnRebalDate / daaDataOnRebalDate.shift(3) - 1
    momentum6 = daaDataOnRebalDate / daaDataOnRebalDate.shift(6) -1
    momentum12 = daaDataOnRebalDate / daaDataOnRebalDate.shift(12) -1

    momentumScore = 12*momentum1 + 4*momentum3 + 2*momentum6 + momentum12
    # ========================================================================
    # 5. 모멘텀 상세 정보 출력 (옵션)
    # ========================================================================
    if show_momentum_details:
        print("\n" + "="*70)
        print("모멘텀 스코어 계산 공식:")
        print("최종 스코어 = 12 × (1개월 수익률) + 4 × (3개월 수익률) + 2 × (6개월 수익률) + 1 × (12개월 수익률)")
        print("="*70)
        
        # print("\n[1개월 모멘텀 (가중치: 12)]")
        # print(momentum1.tail(3))
        
        # print("\n[3개월 모멘텀 (가중치: 4)]")
        # print(momentum3.tail(3))
        
        # print("\n[6개월 모멘텀 (가중치: 2)]")
        # print(momentum6.tail(3))
        
        # print("\n[12개월 모멘텀 (가중치: 1)]")
        # print(momentum12.tail(3))
        
        print("\n[최종 가중 모멘텀 스코어]")
        print(momentumScore.tail(3))
        print("="*70 + "\n")
    
    # 결측치 제거
    momentumScore.dropna(inplace=True)

    # 카나리아 자산 모멘텀 스코어가 모두 0 초과일 때 공격자산 6개 전체에 균등배분 (각 16.67%)
    # 카나리아 자산 중 하나의 자산만 모멘텀 스코어0 초과일 때 모멘텀 스코어가 가장 큰 공격 자산 1개와 방어자산 1개
    # 카나리아 자산 모두 모멘텀스코어가 0 이하라면 수비 자산 중 모멘텀 스코어가 가장 큰 자산에 몰빵
    isAttack = (momentumScore[daaCanary] > 0).sum(axis=1)
    daaWeight = momentumScore.apply(applyGetDAAWeight_by_weighted_avg, axis=1, args=(isAttack,))
    return daaWeight


def applyGetDAAWeight_by_weighted_avg(row, isAttack):
    daaCol = ['SMH', 'CQQQ', 'SOXX', 'SPY', 'IEMG', 'EWY','DX-Y.NYB','GLD', 'TLT', 'VWO','BND',"LQD"]
    daaAttack = ['SMH', 'CQQQ', 'SOXX', 'SPY', 'IEMG', 'EWY']
    daaDefense = ['DX-Y.NYB', 'GLD', 'TLT']
    daaCanary = ["VWO", "BND"]

    # 결과 Series 초기화 (모든 자산 가중치 0으로 시작)
    result = pd.Series(0.0, index=row.index, name=row.name)
    
    # 카나리아 2개 살아있을 때 - 공격자산 100%를 모멘텀 비례 배분
    if isAttack[row.name] == 2:
        # 공격자산의 모멘텀 스코어 합계 계산
        attack_scores = row[daaAttack]
        total_attack_score = attack_scores.sum()
        
        # 각 공격자산에 모멘텀 스코어 비례로 가중치 배분
        # 예: SMH 가중치 = SMH 스코어 / 전체 공격자산 스코어 합계
        if total_attack_score > 0:
            result[daaAttack] = attack_scores / total_attack_score
        else:
            # 모든 모멘텀이 음수인 경우 균등 배분
            result[daaAttack] = 1.0 / len(daaAttack)
        
        return result
    
    # 카나리아 1개 살아있을 때: 공격자산 50%, 방어자산 50%
    if isAttack[row.name] == 1:
        # 공격자산 50% 배분
        attack_scores = row[daaAttack]
        total_attack_score = attack_scores.sum()
        
        if total_attack_score > 0:
            # 각 공격자산에 전체 포트폴리오의 50% 중에서 모멘텀 비례로 배분
            result[daaAttack] = 0.5 * (attack_scores / total_attack_score)
        else:
            # 모든 모멘텀이 음수인 경우 균등 배분
            result[daaAttack] = 0.5 / len(daaAttack)
        
        # 방어자산 50% 배분
        defense_scores = row[daaDefense]
        total_defense_score = defense_scores.sum()
        
        if total_defense_score > 0:
            result[daaDefense] = 0.5 * (defense_scores / total_defense_score)
        else:
            # 모든 모멘텀이 음수인 경우 균등 배분
            result[daaDefense] = 0.5 / len(daaDefense)
        
        return result
    
    # 카나리아 0개: 방어자산 전체를 동일비중으로 보유
    # 방어자산의 모멘텀 스코어 합계 계산
    defense_scores = row[daaDefense]
    total_defense_score = defense_scores.sum()

    if total_defense_score > 0:
        result[daaDefense] = defense_scores / total_defense_score
    else:
        # 모든 모멘텀이 음수인 경우 균등 배분
        result[daaDefense] = 1.0 / len(daaDefense)

    return result


## 카나리
"A":    daaCanary = ["VWO", "BND"]      # Original

"B":    daaCanary = ["LQD", "BND"]      # 신용 위험 중심

"C":    daaCanary = ["SPY", "BND"]      # 미국 시장 중심

In [114]:
daaWeight = getDAAWeight_wa(close_data, 'A').loc["2019-12-31":]
daaDayReturn, daaCumReturn_a = cal_rebalancing_portfolio(close_data=close_data, weight_df=daaWeight, enable_plot=False)
daaCagr, daaDD_a, daaMDD = evaluate_performance(daaDayReturn, daaCumReturn_a)

▶ 최종 수익률   : 141.20%
=== 성과 평가 리포트 ===
1. 수익성 (CAGR) : 15.68%
2. 안정성 (MDD)  : -33.21%
------------------------------
3. 연간 변동성   : 28.62%
4. 샤프 지수     : 0.5822


In [115]:
daaWeight = getDAAWeight_wa(close_data, 'B').loc["2019-12-31":]
daaDayReturn, daaCumReturn_a = cal_rebalancing_portfolio(close_data=close_data, weight_df=daaWeight, enable_plot=False)
daaCagr, daaDD_a, daaMDD = evaluate_performance(daaDayReturn, daaCumReturn_a)

▶ 최종 수익률   : 123.53%
=== 성과 평가 리포트 ===
1. 수익성 (CAGR) : 14.24%
2. 안정성 (MDD)  : -43.00%
------------------------------
3. 연간 변동성   : 41.10%
4. 샤프 지수     : 0.4802


In [116]:
daaWeight = getDAAWeight_wa(close_data, 'C').loc["2019-12-31":]
daaDayReturn, daaCumReturn_a = cal_rebalancing_portfolio(close_data=close_data, weight_df=daaWeight, enable_plot=False)
daaCagr, daaDD_a, daaMDD = evaluate_performance(daaDayReturn, daaCumReturn_a)

▶ 최종 수익률   : 291.18%
=== 성과 평가 리포트 ===
1. 수익성 (CAGR) : 25.32%
2. 안정성 (MDD)  : -165.56%
------------------------------
3. 연간 변동성   : 571.17%
4. 샤프 지수     : -0.5190


# 3. weighted Average - SPY 제거 QQQ 버전


In [70]:
def getDAAWeight_QQQ(closeDataSet, case='A',  show_momentum_details=False):

    daaCol = ['SMH', 'CQQQ', 'SOXX', 'SPY', 'IEMG', 'EWY','DX-Y.NYB','GLD', 'TLT', 'VWO', 'BND',"LQD", 'QQQ']
    daaAttack = ['SMH', 'CQQQ', 'SOXX', 'QQQ', 'IEMG', 'EWY']

    daaDefense = ['DX-Y.NYB', 'GLD', 'TLT']
    
    # Case별 카나리아 자산 설정
    if case == "A":
        daaCanary = ["VWO", "BND"]      # Original
    elif case == "B":
        daaCanary = ["LQD", "BND"]      # 신용 위험 중심
    elif case == "C":
        daaCanary = ["SPY", "BND"]      # 미국 시장 중심
    else:
        daaCanary = ["VWO", "BND"]      # Default

    daaData = closeDataSet[daaCol].copy()
    daaData.dropna(inplace=True)

    # 모멘텀 스코어 계산
    rebalDate = get_rebalancing_dates(daaData)
    daaDataOnRebalDate = daaData.loc[rebalDate]

    momentum1 = daaDataOnRebalDate / daaDataOnRebalDate.shift(1) -1
    momentum3 = daaDataOnRebalDate / daaDataOnRebalDate.shift(3) - 1
    momentum6 = daaDataOnRebalDate / daaDataOnRebalDate.shift(6) -1
    momentum12 = daaDataOnRebalDate / daaDataOnRebalDate.shift(12) -1

    momentumScore = 12*momentum1 + 4*momentum3 + 2*momentum6 + momentum12
    # ========================================================================
    # 5. 모멘텀 상세 정보 출력 (옵션)
    # ========================================================================
    if show_momentum_details:
        print("\n" + "="*70)
        print("모멘텀 스코어 계산 공식:")
        print("최종 스코어 = 12 × (1개월 수익률) + 4 × (3개월 수익률) + 2 × (6개월 수익률) + 1 × (12개월 수익률)")
        print("="*70)
        
        # print("\n[1개월 모멘텀 (가중치: 12)]")
        # print(momentum1.tail(3))
        
        # print("\n[3개월 모멘텀 (가중치: 4)]")
        # print(momentum3.tail(3))
        
        # print("\n[6개월 모멘텀 (가중치: 2)]")
        # print(momentum6.tail(3))
        
        # print("\n[12개월 모멘텀 (가중치: 1)]")
        # print(momentum12.tail(3))
        
        print("\n[최종 가중 모멘텀 스코어]")
        print(momentumScore.tail(3))
        print("="*70 + "\n")
    
    # 결측치 제거
    momentumScore.dropna(inplace=True)

    # 카나리아 자산 모멘텀 스코어가 모두 0 초과일 때 공격자산 6개 전체에 균등배분 (각 16.67%)
    # 카나리아 자산 중 하나의 자산만 모멘텀 스코어0 초과일 때 모멘텀 스코어가 가장 큰 공격 자산 1개와 방어자산 1개
    # 카나리아 자산 모두 모멘텀스코어가 0 이하라면 수비 자산 중 모멘텀 스코어가 가장 큰 자산에 몰빵
    isAttack = (momentumScore[daaCanary] > 0).sum(axis=1)
    daaWeight = momentumScore.apply(applyGetDAAWeight_by_weighted_avg_QQQ, axis=1, args=(isAttack,))
    return daaWeight


def applyGetDAAWeight_by_weighted_avg_QQQ(row, isAttack):
    daaCol = ['SMH', 'CQQQ', 'SOXX', 'SPY', 'IEMG', 'EWY','DX-Y.NYB','GLD', 'TLT', 'VWO', 'BND',"LQD", 'QQQ']
    daaAttack = ['SMH', 'CQQQ', 'SOXX', 'QQQ', 'IEMG', 'EWY']
    daaDefense = ['DX-Y.NYB', 'GLD', 'TLT']
    daaCanary = ["VWO", "BND"]

    # 결과 Series 초기화 (모든 자산 가중치 0으로 시작)
    result = pd.Series(0.0, index=row.index, name=row.name)
    
    # 카나리아 2개 살아있을 때 - 공격자산 100%를 모멘텀 비례 배분
    if isAttack[row.name] == 2:
        # 공격자산의 모멘텀 스코어 합계 계산
        attack_scores = row[daaAttack]
        total_attack_score = attack_scores.sum()
        
        # 각 공격자산에 모멘텀 스코어 비례로 가중치 배분
        # 예: SMH 가중치 = SMH 스코어 / 전체 공격자산 스코어 합계
        if total_attack_score > 0:
            result[daaAttack] = attack_scores / total_attack_score
        else:
            # 모든 모멘텀이 음수인 경우 균등 배분
            result[daaAttack] = 1.0 / len(daaAttack)
        
        return result
    
    # 카나리아 1개 살아있을 때: 공격자산 50%, 방어자산 50%
    if isAttack[row.name] == 1:
        # 공격자산 50% 배분
        attack_scores = row[daaAttack]
        total_attack_score = attack_scores.sum()
        
        if total_attack_score > 0:
            # 각 공격자산에 전체 포트폴리오의 50% 중에서 모멘텀 비례로 배분
            result[daaAttack] = 0.5 * (attack_scores / total_attack_score)
        else:
            # 모든 모멘텀이 음수인 경우 균등 배분
            result[daaAttack] = 0.5 / len(daaAttack)
        
        # 방어자산 50% 배분
        defense_scores = row[daaDefense]
        total_defense_score = defense_scores.sum()
        
        if total_defense_score > 0:
            result[daaDefense] = 0.5 * (defense_scores / total_defense_score)
        else:
            # 모든 모멘텀이 음수인 경우 균등 배분
            result[daaDefense] = 0.5 / len(daaDefense)
        
        return result
    
    # 카나리아 0개: 방어자산 전체를 동일비중으로 보유
    # 방어자산의 모멘텀 스코어 합계 계산
    defense_scores = row[daaDefense]
    total_defense_score = defense_scores.sum()

    if total_defense_score > 0:
        result[daaDefense] = defense_scores / total_defense_score
    else:
        # 모든 모멘텀이 음수인 경우 균등 배분
        result[daaDefense] = 1.0 / len(daaDefense)

    return result


## 카나리
"A":    daaCanary = ["VWO", "BND"]      # Original

"B":    daaCanary = ["LQD", "BND"]      # 신용 위험 중심

"C":    daaCanary = ["SPY", "BND"]      # 미국 시장 중심

In [103]:
daaWeight = getDAAWeight_QQQ(close_data, 'A').loc["20-12-31":]
daaDayReturn, daaCumReturn_a = cal_rebalancing_portfolio(close_data=close_data, weight_df=daaWeight, enable_plot=False)
daaCagr, daaDD_a, daaMDD = evaluate_performance(daaDayReturn, daaCumReturn_a)

▶ 최종 수익률   : 259.33%
=== 성과 평가 리포트 ===
1. 수익성 (CAGR) : 23.57%
2. 안정성 (MDD)  : -51.17%
------------------------------
3. 연간 변동성   : 51.77%
4. 샤프 지수     : 0.6125


In [104]:
daaWeight = getDAAWeight_QQQ(close_data, 'B').loc["2019-12-31":]
daaDayReturn, daaCumReturn_a = cal_rebalancing_portfolio(close_data=close_data, weight_df=daaWeight, enable_plot=False)
daaCagr, daaDD_a, daaMDD = evaluate_performance(daaDayReturn, daaCumReturn_a)

▶ 최종 수익률   : 368.09%
=== 성과 평가 리포트 ===
1. 수익성 (CAGR) : 29.10%
2. 안정성 (MDD)  : -62.29%
------------------------------
3. 연간 변동성   : 82.90%
4. 샤프 지수     : 0.6489


In [105]:
daaWeight = getDAAWeight_QQQ(close_data, 'C').loc["2019-12-31":]
daaDayReturn, daaCumReturn_a = cal_rebalancing_portfolio(close_data=close_data, weight_df=daaWeight, enable_plot=False)
daaCagr, daaDD_a, daaMDD = evaluate_performance(daaDayReturn, daaCumReturn_a)

▶ 최종 수익률   : 320.68%
=== 성과 평가 리포트 ===
1. 수익성 (CAGR) : 26.84%
2. 안정성 (MDD)  : -51.17%
------------------------------
3. 연간 변동성   : 56.62%
4. 샤프 지수     : 0.6526


# 4. 모멘텀 + 리스크패러티 weight

In [139]:
import pandas as pd
import numpy as np

# 글로벌 변수 (함수 내 재정의 제거)
DAA_COL = ['SMH', 'CQQQ', 'SOXX', 'SPY', 'IEMG', 'EWY','DX-Y.NYB','GLD', 'TLT', 'VWO','BND',"LQD"]
DAA_ATTACK = ['SMH', 'CQQQ', 'SOXX', 'SPY', 'IEMG', 'EWY']
DAA_DEFENSE = ['DX-Y.NYB', 'GLD', 'TLT']

def compute_vol_df(close_df, window=60):
    """수익률로부터 최근 window일 vol 계산"""
    ret_df = close_df[DAA_COL].pct_change()
    vol = ret_df.rolling(window=window).std() * np.sqrt(252)  # 연환산
    return vol

def risk_parity_weights_inverse_vol(vol_row, tickers):
    """인버스 볼 기반 리스크패러티 weight (각 버킷 내)"""
    vols = vol_row[tickers].replace(0, np.nan)
    inv_vols = 1.0 / vols.fillna(vols.median() or 0.2)  # 중앙값으로 대체
    inv_vols = inv_vols.replace([np.inf, -np.inf], 0.0)
    total_inv = inv_vols.sum()
    return pd.Series(inv_vols / total_inv if total_inv > 0 else 
                     1.0/len(tickers), index=tickers)

def apply_momentum_rp_filter(scores, rp_weights):
    """모멘텀 > 0 필터링 후 RP weight 재정규화"""
    mask_pos = scores > 0
    if mask_pos.any():
        w_tmp = rp_weights.copy()
        w_tmp[~mask_pos] = 0.0
        s = w_tmp.sum()
        return w_tmp / s if s > 0 else rp_weights
    return rp_weights  # 모두 음수면 RP 그대로

def applyGetDAAWeight_rp(row, attack_level, vol_row):  # isAttack -> attack_level (int)
    """attack_level이 int로 직접 전달되도록"""
    result = pd.Series(0.0, index=DAA_COL, name=row.name)
    
    w_attack_rp = risk_parity_weights_inverse_vol(vol_row, DAA_ATTACK)
    w_defense_rp = risk_parity_weights_inverse_vol(vol_row, DAA_DEFENSE)
    
    if attack_level == 2:
        result[DAA_ATTACK] = apply_momentum_rp_filter(row[DAA_ATTACK], w_attack_rp)
    elif attack_level == 1:
        result[DAA_ATTACK] = 0.5 * apply_momentum_rp_filter(row[DAA_ATTACK], w_attack_rp)
        result[DAA_DEFENSE] = 0.5 * apply_momentum_rp_filter(row[DAA_DEFENSE], w_defense_rp)
    else:
        result[DAA_DEFENSE] = apply_momentum_rp_filter(row[DAA_DEFENSE], w_defense_rp)
    
    return result


def getDAAWeight_mixed(closeDataSet, case='A', show_momentum_details=False, vol_window=60):
    """
    리스크패러티 버전
    """
    
    # Case별 카나리아
    if case == "A": 
        daaCanary = ["VWO", "BND"]      # Original
    elif case == "B": 
        daaCanary = ["LQD", "BND"]      # 신용 위험 중심
    elif case == "C": 
        daaCanary = ["SPY", "BND"]      # 미국 시장 중심
    else: 
        daaCanary = ["VWO", "BND"]
    
    daaData = closeDataSet[DAA_COL].copy().dropna()
    rebalDate = get_rebalancing_dates(daaData)
    daaDataOnRebalDate = daaData.loc[rebalDate]
    
    # 모멘텀 계산 (기존)
    momentum1 = daaDataOnRebalDate / daaDataOnRebalDate.shift(1) - 1
    momentum3 = daaDataOnRebalDate / daaDataOnRebalDate.shift(3) - 1
    momentum6 = daaDataOnRebalDate / daaDataOnRebalDate.shift(6) - 1
    momentum12 = daaDataOnRebalDate / daaDataOnRebalDate.shift(12) - 1
    momentumScore = 12*momentum1 + 4*momentum3 + 2*momentum6 + momentum12
    momentumScore.dropna(inplace=True)
    
    if show_momentum_details:
        print("\n최종 모멘텀 스코어 (최근 3개):\n", momentumScore.tail(3))
    
    isAttack = (momentumScore[daaCanary] > 0).sum(axis=1)
    
    # Vol 계산
    vol_df = compute_vol_df(closeDataSet, vol_window)
    
    # 각 rebal date별 전체 momentum row 전달
    daaWeight = []
    for date in momentumScore.index:
        vol_row = vol_df.loc[date] if date in vol_df.index else \
                  vol_df.iloc[vol_df.index.get_indexer([date], method='nearest')]
        
        weight_row = applyGetDAAWeight_rp(
            momentumScore.loc[date],  # Series 
            isAttack.loc[date],       # int (0,1,2) 
            vol_row                   # Series 
        )
        daaWeight.append(weight_row)
    
    return pd.DataFrame(daaWeight, index=momentumScore.index)


# 사용법
# weights = getDAAWeight_mixed(close_prices, case='A', vol_window=60, show_momentum_details=True)


In [140]:
daaWeight = getDAAWeight_mixed(close_data, 'A', vol_window=60, show_momentum_details=False).loc["2019-12-31":]
daaDayReturn, daaCumReturn_a = cal_rebalancing_portfolio(close_data=close_data, weight_df=daaWeight, enable_plot=False)
daaCagr, daaDD_a, daaMDD = evaluate_performance(daaDayReturn, daaCumReturn_a)

▶ 최종 수익률   : 104.31%
=== 성과 평가 리포트 ===
1. 수익성 (CAGR) : 12.55%
2. 안정성 (MDD)  : -20.56%
------------------------------
3. 연간 변동성   : 16.02%
4. 샤프 지수     : 0.6934


In [135]:
daaWeight = getDAAWeight_mixed(close_data, 'B', vol_window=60, show_momentum_details=False).loc["2019-12-31":]
daaDayReturn, daaCumReturn_a = cal_rebalancing_portfolio(close_data=close_data, weight_df=daaWeight, enable_plot=False)
daaCagr, daaDD_a, daaMDD = evaluate_performance(daaDayReturn, daaCumReturn_a)

▶ 최종 수익률   : 83.72%
=== 성과 평가 리포트 ===
1. 수익성 (CAGR) : 10.59%
2. 안정성 (MDD)  : -30.08%
------------------------------
3. 연간 변동성   : 18.87%
4. 샤프 지수     : 0.5226


In [136]:
daaWeight = getDAAWeight_mixed(close_data, 'C', vol_window=60, show_momentum_details=False).loc["2019-12-31":]
daaDayReturn, daaCumReturn_a = cal_rebalancing_portfolio(close_data=close_data, weight_df=daaWeight, enable_plot=False)
daaCagr, daaDD_a, daaMDD = evaluate_performance(daaDayReturn, daaCumReturn_a)

▶ 최종 수익률   : 111.04%
=== 성과 평가 리포트 ===
1. 수익성 (CAGR) : 13.15%
2. 안정성 (MDD)  : -21.12%
------------------------------
3. 연간 변동성   : 16.82%
4. 샤프 지수     : 0.7003
